# HydroSeason - Global Product Fetch Example

End-to-end workflow for catchments **without** an existing rainfall record:

1. Load a catchment polygon from a GeoJSON file
2. Fetch spatially averaged monthly rainfall from the global product path: CHIRPS v3 first, ERA5 as backup when configured
3. Optionally force ERA5 exact mode for final reporting or comparison
4. Run the HydroSeason pipeline on the fetched data
5. Visualise and export results

> **Requirements:** this notebook installs the local checkout with `fetch` and `plot` extras.  
> Internet access is required for remote gridded products. Results are cached locally to avoid re-downloading.


In [ ]:
import sys
from pathlib import Path

_repo_root = Path("..").resolve()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))


In [ ]:
# Install HydroSeason with fetch and plotting extras from this checkout.
%pip install -e "..[fetch]" -q


## 1 — Imports


In [ ]:
import pandas as pd
import geopandas as gpd

from hydroseason import classify_rainfall, get_monthly_aoi_rainfall
from hydroseason.report import display_summary, generate_html_report
from hydroseason.plot import (
    plot_season_timeline,
    plot_agg_monthly_rainfall,
    plot_annual_metrics,
    plot_dashboard,
)


## 2 — Load the catchment polygon

The repository ships with `data/fitzroy_catchment.geojson` — an approximate bounding polygon for the Fitzroy River catchment in NW Australia — as a ready-to-run demonstration area.

Swap `GEOJSON_PATH` for any other polygon (Shapefile, GeoJSON, GeoPackage) to analyse a different catchment.


In [ ]:
GEOJSON_PATH = Path("../data/fitzroy_catchment.geojson")

gdf = gpd.read_file(GEOJSON_PATH)
print(f"CRS : {gdf.crs}")
print(f"Bounds: {gdf.total_bounds.round(4)}")
print(f"Name  : {gdf['name'].iloc[0]}")
gdf


## 3 - Fetch CHIRPS-first monthly rainfall

For global rainfall, prefer `get_monthly_aoi_rainfall(..., source="chirps")`. It reads CHIRPS v3 monthly rasters directly, which avoids ERA5 hourly-to-monthly aggregation. Supplying `era5_zarr_path` lets ERA5 fill gaps CHIRPS cannot cover, such as pre-1981 years, locations outside 60S-60N, or unavailable recent months.

For Australian production runs, use `source="auto"` so SILO remains the default. This notebook uses `source="chirps"` to demonstrate the global product path even though the bundled Fitzroy polygon is in Australia.

Key parameters:
| Parameter | Description |
|---|---|
| `source` | `"chirps"` for global CHIRPS-first, `"auto"` for SILO-in-Australia / CHIRPS-elsewhere |
| `era5_zarr_path` | Optional ERA5 backup URI |
| `gdf` | GeoDataFrame with the catchment polygon |
| `start_year` / `end_year` | Temporal range (inclusive) |
| `cache_dir` | Local folder to cache final monthly results |


In [ ]:
# ERA5 public Zarr store on Google Cloud Storage - used only as backup here.
ERA5_ZARR = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

START_YEAR = 1985
END_YEAR = 2024
FETCH_SOURCE = "chirps"  # Use "auto" to prefer SILO for Australian AOIs.
SHOW_FETCH_PROGRESS = True

CACHE_DIR = Path("../data/fetch_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

rainfall_df = get_monthly_aoi_rainfall(
    gdf,
    start_year=START_YEAR,
    end_year=END_YEAR,
    source=FETCH_SOURCE,
    era5_zarr_path=ERA5_ZARR,
    cache_dir=CACHE_DIR,
    show_progress=SHOW_FETCH_PROGRESS,
)

print(
    f"Fetched {len(rainfall_df)} monthly records "
    f"({rainfall_df['Date'].min()} -> {rainfall_df['Date'].max()})"
)
display(rainfall_df.head())
display(rainfall_df[["Data_Source", "Data_Product"]].drop_duplicates())


In [ ]:
out_csv = Path("../data/fitzroy_global_product_rainfall.csv")
rainfall_df.to_csv(out_csv, index=False)
print(f"Saved -> {out_csv.resolve()}")


## 4 - Run the HydroSeason pipeline

`classify_rainfall` accepts any DataFrame with columns `[Date, Year, Month, <value>]`. The `Rainfall_mm` column produced by the AOI fetch wrapper matches the default `value_col` directly. Fetch metadata columns such as `Data_Source` are carried through so mixed CHIRPS/ERA5 series remain auditable.


In [ ]:
artifacts = classify_rainfall(rainfall_df)
result = artifacts.result

display_summary(artifacts)


## 5 — Visualise results


In [ ]:
# Season timeline — each month coloured by SeasonType, hydro-year boundaries marked
plot_season_timeline(result)


In [ ]:
# Aggregated monthly rainfall - mean rainfall per calendar month coloured by baseline season
plot_agg_monthly_rainfall(result, artifacts.fixed_monthly)


In [ ]:
# Stacked wet/dry totals per hydrological year + wet month count
plot_annual_metrics(result)


In [ ]:
# Composite dashboard: timeline + climatology + annual totals in one figure
plot_dashboard(artifacts)


## 6 - Export results

Save the delineated season table as a CSV and generate a self-contained HTML report.


In [ ]:
# Season delineation table
result_csv = Path("../data/fitzroy_global_product_hydroseason.csv")
result.to_csv(result_csv, index=False)
print(f"Results -> {result_csv.resolve()}")

# Self-contained HTML report (no Python needed to view)
report_path = generate_html_report(artifacts, "hydroseason_global_product_report.html")
print(f"Report  -> {report_path.resolve()}")


## Optional - Force ERA5 Exact Mode

Use this only when you need the hourly ERA5 product specifically, for example as a final comparison or publication-quality exact run. It is slower than the CHIRPS-first path.


In [ ]:
RUN_ERA5_EXACT = False

if RUN_ERA5_EXACT:
    era5_exact_df = get_monthly_aoi_rainfall(
        gdf,
        start_year=START_YEAR,
        end_year=END_YEAR,
        source="era5",
        era5_zarr_path=ERA5_ZARR,
        variable="rainfall",
        cache_dir=Path("../data/era5_cache"),
        spatial_chunk="auto",
        time_chunk="auto",
        temporal_batch_years="auto",
        show_progress=SHOW_FETCH_PROGRESS,
    )
    era5_exact_df.to_csv(Path("../data/fitzroy_era5_exact_rainfall.csv"), index=False)
    display(era5_exact_df.head())
    display(era5_exact_df[["Data_Source", "Data_Product"]].drop_duplicates())
else:
    print("Set RUN_ERA5_EXACT = True to force the slower ERA5 exact path.")
